## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from av2.datasets.motion_forecasting import scenario_serialization
from av2.datasets.motion_forecasting.eval import metrics
import tqdm
import torch
from torch import nn

## Load data

In [ ]:
backbone = "DeMo"
program_name = "simplistic"

In [ ]:
data_root = Path("../data")
val_ground_truth_path = data_root / "av2" / "val_ground_truth.parquet"
test_ground_truth_path = data_root / "av2" / "av2_mf_focal_test_annotations.parquet"
test_prediction_path = data_root / "av2_backbone_outputs" / backbone / "submission_test.parquet"
train_data_path = data_root / "results" / program_name / "val" / backbone / "scored_predictions.parquet"
test_data_path = data_root / "tracs_predictions" / program_name / backbone / "test.parquet"
features_pt_path = data_root / "results" / program_name / "val" / backbone / "features.pt"
optimal_weights_pt_path = data_root / "results" / program_name / "val" / backbone / "optimal_weights.pt"

## Run tests

In [ ]:
def get_val_gt_trajectory(scenario_id: str) -> pd.DataFrame:
    scenario_path = data_root / "av2" / "val" / scenario_id / f"scenario_{scenario_id}.parquet"
    scenario = scenario_serialization.load_argoverse_scenario_parquet(scenario_path)
    focal_track = next(t for t in scenario.tracks if t.track_id == scenario.focal_track_id)
    future_states = [s for s in focal_track.object_states if not s.observed]
    gt = np.array([[s.position[0], s.position[1]] for s in future_states])
    return pd.DataFrame({
        "scenario_id": [scenario_id],
        "gt_trajectory_x": [gt[:, 0].tolist()],
        "gt_trajectory_y": [gt[:, 1].tolist()],
    })

In [ ]:
src = data_root / "tracs_predictions" / program_name / backbone / "val.parquet"
train_data = pd.read_parquet(src).drop(columns=[
    "agent_type", "covariance", "is_moving", "min_dist_to_focal",
    "included_in_features", "tracs_backbone_independent_op",
    "tracs_backbone_linear_op", "tracs_pointwise_scores",
])
train_data_path.parent.mkdir(parents=True, exist_ok=True)
train_data.to_parquet(train_data_path)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

if not val_ground_truth_path.exists():
    scenario_ids = [p.name for p in (data_root / "av2" / "val").iterdir() if p.is_dir()]
    with ThreadPoolExecutor(32) as pool:
        rows = list(tqdm.tqdm(pool.map(get_val_gt_trajectory, scenario_ids), total=len(scenario_ids)))
    pd.concat(rows, ignore_index=True).to_parquet(val_ground_truth_path)

val_gt = pd.read_parquet(val_ground_truth_path).set_index("scenario_id")

## Fusion Network Architecture

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 2)           # w1, w2 are unconstrained
        )

    @staticmethod
    def extract_features(predicted_tracks):
        relative_tracks = predicted_tracks - predicted_tracks[:, :1, :]  # (6, 60, 2)
        scale = relative_tracks.std() + 1e-6
        return (relative_tracks / scale).flatten().astype(np.float32)  # (720,)

    def forward(self, x):
        return self.net(x)

    def predict_weights(self, predicted_tracks):
        x = torch.tensor(FusionNet.extract_features(predicted_tracks)).unsqueeze(0).to(next(self.parameters()).device)
        with torch.no_grad():
            return self.forward(x).squeeze(0).cpu().numpy()

## Supervisor

In [ ]:
def optimal_weights_grid(log_tracs_scores, log_backbone_scores, predicted_tracks, gt_track, n_grid=25):
    """Vectorized grid search over (w1, w2)."""

    w = np.linspace(0, 1, n_grid)
    W1, W2 = np.meshgrid(w, w, indexing="ij")
    W1, W2 = W1.flatten(), W2.flatten()

    log_fused = W1[:, None] * log_tracs_scores + W2[:, None] * log_backbone_scores
    log_fused -= log_fused.max(axis=1, keepdims=True)
    fused = np.exp(log_fused)
    fused /= fused.sum(axis=1, keepdims=True)  # (n_grid², K)

    # brier_ade = ade + (1-p)²  — vectorized over all weight combos at once
    ade_vector = metrics.compute_ade(predicted_tracks, gt_track)  # (K,)
    brier_ade = ade_vector + np.square(1 - fused)  # (n_grid², K)

    raw_backbone_score = (ade_vector + np.square(1 - np.exp(log_backbone_scores))).min()
    raw_tracs_score = (ade_vector + np.square(1 - np.exp(log_tracs_scores))).min()

    idx = np.argmin(brier_ade.min(axis=1))
    optimized_score = brier_ade[idx].min()

    return W1[idx], W2[idx], raw_backbone_score, raw_tracs_score, optimized_score

## Training

In [ ]:
train_data = pd.read_parquet(train_data_path)

In [ ]:
if not features_pt_path.exists() or not optimal_weights_pt_path.exists():
    all_features = []
    all_weights = []

    for scenario_id, scenario_data in tqdm.tqdm(train_data.groupby("scenario_id")):
        gt_row = val_gt.loc[scenario_id]
        gt_track = np.stack([
            np.array(gt_row["gt_trajectory_x"]),
            np.array(gt_row["gt_trajectory_y"])
        ], axis=1).astype(np.float64)

        log_tracs_scores = np.log(np.array(scenario_data["tracs_trajectory_score"]))
        log_backbone_scores = np.log(np.array(scenario_data["backbone_trajectory_score"]))
        predicted_tracks = np.moveaxis(
            np.array([[*scenario_data["x_traj"]], [*scenario_data["y_traj"]]]),
            source=[0, 1, 2], destination=[2, 0, 1],
        ).astype(np.float64)

        w1, w2, _, _, _ = optimal_weights_grid(log_tracs_scores, log_backbone_scores, predicted_tracks, gt_track)

        all_features.append(FusionNet.extract_features(predicted_tracks))
        all_weights.append([w1, w2])

    F = torch.tensor(np.stack(all_features), dtype=torch.float32)
    W = torch.tensor(np.array(all_weights), dtype=torch.float32)
    print(f"Dataset: {F.shape} features, {W.shape} labels")

    torch.save(F, features_pt_path)
    torch.save(W, optimal_weights_pt_path)

In [ ]:
F = torch.load(features_pt_path)
W = torch.load(optimal_weights_pt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
F, W = F.to(device), W.to(device)

n_epochs = 200
model = FusionNet(F.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

for epoch in range(n_epochs):
    for idx in torch.randperm(len(F)).split(256):
        optimizer.zero_grad()
        nn.functional.mse_loss(model(F[idx]), W[idx]).backward()
        optimizer.step()
    scheduler.step()
    if epoch % 20 == 0:
        with torch.no_grad():
            loss = nn.functional.mse_loss(model(F), W).item()
            w_pred = model(F).cpu().numpy()
            print(f"Epoch {epoch}: loss={loss:.4f}  mean w1={w_pred[:,0].mean():.2f}  mean w2={w_pred[:,1].mean():.2f}")

In [ ]:
test_data = pd.read_parquet(test_data_path)
ground_truth = pd.read_parquet(test_ground_truth_path)
backbone_scores = []
fusion_scores = []

for scenario_id in tqdm.tqdm(test_data["scenario_id"].unique()):
    gt_row = ground_truth[ground_truth["scenario_id"] == scenario_id]
    gt_track = np.stack([
        np.array(gt_row["gt_trajectory_x"].iloc[0]),
        np.array(gt_row["gt_trajectory_y"].iloc[0])
    ], axis=1).astype(np.float64)

    scenario_data = test_data[test_data["scenario_id"] == scenario_id]
    backbone_probs = np.array(scenario_data["backbone_trajectory_score"], dtype=np.float64)
    tracs_scores = np.array(scenario_data["tracs_trajectory_score"], dtype=np.float64)
    predicted_tracks = np.moveaxis(
        np.array([[*scenario_data["x_traj"]], [*scenario_data["y_traj"]]]),
        source=[0, 1, 2], destination=[2, 0, 1],
    ).astype(np.float64)

    # Backbone only
    backbone_probs_norm = backbone_probs / backbone_probs.sum()
    backbone_scores.append(metrics.compute_brier_ade(predicted_tracks, gt_track, backbone_probs_norm).min())

    # FusionNet
    w1, w2 = model.predict_weights(predicted_tracks)
    log_fused = w1 * np.log(tracs_scores + 1e-10) + w2 * np.log(backbone_probs + 1e-10)
    fused_probs = np.exp(log_fused - log_fused.max())
    fused_probs /= fused_probs.sum()
    fusion_scores.append(metrics.compute_brier_ade(predicted_tracks, gt_track, fused_probs).min())

backbone_scores = np.array(backbone_scores)
fusion_scores = np.array(fusion_scores)

print(f"Backbone   Brier-minADE: {backbone_scores.mean():.4f}")
print(f"FusionNet  Brier-minADE: {fusion_scores.mean():.4f}")
print(f"Delta: {fusion_scores.mean() - backbone_scores.mean():.4f}  ({(fusion_scores.mean() / backbone_scores.mean() - 1) * 100:.1f}%)")

# Per-scenario improvement distribution
delta = fusion_scores - backbone_scores
print(f"\nImproved: {(delta < 0).mean()*100:.1f}%  Degraded: {(delta > 0).mean()*100:.1f}%")